# Task-Based $GW$ Calculations with **`exciting`**
**by <span style="color:darkgreen">Martí Raya Moreno</span>, <span style="color:darkgreen">Hannah Kleine</span> & <span style="color:darkgreen">Sven Lubeck</span> for [<span style="color:darkgoldenrod">exciting *magnesium*</span>](https://www.exciting-code.org/magnesium)**

<font size="2">Self-contained Jupyter notebook for task-based $GW$ calculations.</font>
<hr style="border:2px solid #DDD"> </hr>


**<span style="color:firebrick">Purpose</span>**: In this tutorial you will learn how a task-based $G_0W_0$ calculation is organized in **`exciting`**. You will run a small silicon example, identify the files produced by each task, and restart only the final quasiparticle-energy task.

The numerical settings are deliberately small. They are suitable for learning the workflow, but not for converged quasiparticle energies.
<hr style="border:2px solid #DDD"> </hr>


<div class="alert alert-block alert-warning">

**Table of Contents**

[0. Before Starting](#Before)  
[1. The Task-Based $GW$ Workflow](#Workflow)  
[2. Preparing the Ground State](#GroundState)  
[3. Running Task-Based $GW$](#RunGW)  
[4. Checking the Result and Restarting](#Restart)  
[5. Optional Workflow Branches](#Optional)  
[6. Summary](#Summary)  
[Literature](#Literature)  

</div>


<a id="Before"></a>


<hr style="border:1px solid #DDD"> </hr>

### <span style="color:#15317E">0. Before Starting</span>

Before running this tutorial, follow the instructions in **`00_before_starting.md`**. Activate the **`excitingjupyter`** virtual environment from the **`exciting`** root directory:

<div style="background-color: rgb(224, 224, 224);">

```bash
source "$EXCITINGROOT/tools/excitingjupyter/venv/excitingvenv/bin/activate"
```

</div>

Keep the notebook name **`task_based_gw.ipynb`**. The helper function used below reads XML blocks from that file.

All numerical input values are in atomic units unless stated otherwise.


<a id="Workflow"></a>


<hr style="border:1px solid #DDD"> </hr>

### <span style="color:#15317E">1. The Task-Based $GW$ Workflow</span>

A $G_0W_0$ calculation starts from a DFT ground state and corrects Kohn-Sham eigenvalues. The self-energy is commonly written as

\begin{equation}
\Sigma \approx i G_0 W_0,
\end{equation}

where $G_0$ is the Kohn-Sham Green function and $W_0$ is the screened Coulomb interaction.

In a task-based calculation, **`exciting`** divides the $GW$ calculation into smaller tasks. Each task writes files. Later tasks read these files. This makes it possible to restart a calculation without repeating every completed step.

This tutorial uses

```xml
<gw taskname="taskGroup">
```


**<span style="color:#15317E">i) Dependencies between tasks</span>**

Read the diagram from top to bottom. A solid arrow shows a dependency used in the example. A dashed arrow shows an optional branch discussed in Section 5.

<figure>

</figure>
<div style="clear: both;"></div>

*Figure 1. Main file dependencies in the task-based $GW$ workflow.*


**<span style="color:#15317E">ii) Direct workflow used in this tutorial</span>**

The direct calculation follows this order:

```text
vxc → Coulomb → sigmax → epsilon → invertEpsilon → sigmac → QPEigenvalues
```

The most important point is the final step:

```text
vxc + sigmax + sigmac → QPEigenvalues
```

Therefore, **`QPEigenvalues`** can be repeated as long as the required **`VXCNN`**, **`SIGMAX`**, and **`SIGMAC`** files are still available and are compatible with the current input.


**<span style="color:#15317E">iii) Main tasks and files</span>**

Every $GW$ task also uses data from the DFT ground-state calculation, such as **`STATE.OUT`**, **`EFERMI.OUT`**, and **`EIGVAL.OUT`**.

task | what the task reads and writes
:--- | :---
**`vxc`** | Computes diagonal exchange-correlation matrix elements. Writes **`VXCNN.OUT`** in binary mode or **`VXCNN.DAT`** in text mode.
**`Coulomb`** | Builds the bare Coulomb matrix and related basis data. Writes **`BARC_Q*.OUT`** and **`SGI_Q*.OUT`**.
**`sigmax`** | Reads the Coulomb files and computes the exchange self-energy. Writes **`SIGMAX_K*.OUT`**.
**`epsilon`** | Reads the Coulomb files and computes the dielectric matrix. Writes **`EPSILON-GW_Q*.OUT`**.
**`invertEpsilon`** | Reads the dielectric matrix and writes **`INVERSE-EPSILON_Q*.OUT`**.
**`sigmac`** | Reads the Coulomb files and the inverse dielectric matrix. Writes **`SIGMAC_K*.OUT`**.
**`QPEigenvalues`** | Reads **`VXCNN.*`**, **`SIGMAX_K*.OUT`**, and **`SIGMAC_K*.OUT`** in the selected format. Always writes both the readable **`EVALQP.DAT`** and binary **`EVALQP.OUT`**.

<div style="clear: both;"></div>

In names such as **`SIGMAC_K*.OUT`**, the symbol **`*`** stands for a point index.

The q-point tasks use

```xml
<qpoints qi="1" qf="-1"/>
```

and the k-point tasks use

```xml
<kpoints ki="1" kf="-1"/>
```

Here, **`1`** means “start with the first point” and **`-1`** means “continue to the last point.”

This tutorial uses **`outputFormat="binary"`**. Binary files are efficient and are the normal choice for restarts. Text output is useful only for small tests.


<a id="GroundState"></a>


<hr style="border:1px solid #DDD"> </hr>

### <span style="color:#15317E">2. Preparing the Ground State</span>

The $GW$ calculation needs a completed DFT ground-state calculation. We use bulk silicon and a small **3 × 3 × 3** k-point grid.

**<span style="color:#15317E">i) Create a run directory</span>**


In [ ]:
%%bash
mkdir -p run_Si_taskGW


For a completely clean rerun, rename or remove an existing **`run_Si_taskGW`** directory before continuing.

**<span style="color:#15317E">ii) Ground-state input</span>**


<span class="SI_TASK_GW_GS"></span>
```xml
<input>

   <title>Silicon: Task-Based GW Ground-State Run</title>

   <structure speciespath="$EXCITINGROOT/species">
      <crystal>
         <basevect>5.13 5.13 0.00</basevect>
         <basevect>5.13 0.00 5.13</basevect>
         <basevect>0.00 5.13 5.13</basevect>
      </crystal>
      <species speciesfile="Si.xml" rmt="2.1">
         <atom coord="0.00 0.00 0.00"/>
         <atom coord="0.25 0.25 0.25"/>
      </species>
   </structure>

   <groundstate
      do="fromscratch"
      rgkmax="7.0"
      ngridk="3 3 3"
      xctype="LDA_PW"
      maxscl="50"/>

</input>
```

Save the notebook if you changed the XML block. The next cell writes the block to **`run_Si_taskGW/input.xml`**.


In [ ]:
from excitingjupyter.utilities import get_input_xml_from_notebook
import xml.etree.ElementTree as ET

input_str = get_input_xml_from_notebook(
    "task_based_gw",
    "SI_TASK_GW_GS",
)
ET.fromstring(input_str)  # Check that the extracted XML is well formed.

with open("./run_Si_taskGW/input.xml", "w") as fid:
    fid.write(input_str)


Replace **`$EXCITINGROOT`** in the input with the path to your local **`exciting`** installation:


In [ ]:
%%bash
cd run_Si_taskGW
python3 -m excitingscripts.setup.excitingroot
cd ..


**<span style="color:#15317E">iii) Run and check the ground state</span>**


In [ ]:
%%bash
cd run_Si_taskGW
time python3 -m excitingscripts.execute.single -f input.xml
cd ..


The following files are needed by the $GW$ step. The last lines of **`INFO.OUT`** should also show that the ground-state run finished normally.


In [ ]:
%%bash
cd run_Si_taskGW
ls -1 INFO.OUT STATE.OUT EFERMI.OUT EIGVAL.OUT
printf "\nLast lines of INFO.OUT:\n"
tail -n 15 INFO.OUT
cd ..


<a id="RunGW"></a>


<hr style="border:1px solid #DDD"> </hr>

### <span style="color:#15317E">3. Running Task-Based $GW$</span>

The ground-state block now uses **`do="skip"`**, because the required DFT files already exist. The **`gw`** element uses **`taskname="taskGroup"`** and lists the seven tasks in the direct workflow.

**<span style="color:#15317E">i) Task-based $GW$ input</span>**


<span class="SI_TASK_GW_FULL"></span>
```xml
<input>

   <title>Silicon: Task-Based GW Run</title>

   <structure speciespath="$EXCITINGROOT/species">
      <crystal>
         <basevect>5.13 5.13 0.00</basevect>
         <basevect>5.13 0.00 5.13</basevect>
         <basevect>0.00 5.13 5.13</basevect>
      </crystal>
      <species speciesfile="Si.xml" rmt="2.1">
         <atom coord="0.00 0.00 0.00"/>
         <atom coord="0.25 0.25 0.25"/>
      </species>
   </structure>

   <groundstate
      do="skip"
      rgkmax="7.0"
      ngridk="3 3 3"
      xctype="LDA_PW"/>

   <gw
      taskname="taskGroup"
      ngridq="2 2 2"
      nempty="12"
      ibgw="1"
      nbgw="8"
      skipgnd="false"
      qdepw="sum"
      coreflag="xal">

      <taskGroup outputFormat="binary">

         <vxc>
            <kpoints ki="1" kf="-1"/>
         </vxc>

         <Coulomb>
            <qpoints qi="1" qf="-1"/>
         </Coulomb>

         <sigmax>
            <kpoints ki="1" kf="-1"/>
         </sigmax>

         <epsilon>
            <qpoints qi="1" qf="-1"/>
         </epsilon>

         <invertEpsilon>
            <qpoints qi="1" qf="-1"/>
         </invertEpsilon>

         <sigmac>
            <kpoints ki="1" kf="-1"/>
         </sigmac>

         <QPEigenvalues>
            <kpoints ki="1" kf="-1"/>
         </QPEigenvalues>

      </taskGroup>

      <mixbasis
         lmaxmb="3"
         epsmb="1.0d-4"
         gmb="0.8"/>

      <barecoul barcevtol="0.5d0"/>
      <scrcoul scrtype="rpa"/>
      <selfenergy method="ac" singularity="mpb"/>
      <freqgrid nomeg="4"/>

   </gw>

</input>
```

**<span style="color:#15317E">ii) Meaning of the main settings</span>**

setting | meaning in this example
:--- | :---
**`taskname="taskGroup"`** | Runs the tasks listed inside **`taskGroup`**.
**`outputFormat="binary"`** | Writes efficient files that later tasks can reuse.
**`ngridq="2 2 2"`** | Sets a small k/q-point grid for the $GW$ calculation.
**`nempty="12"`** | Uses 12 empty states in the screened interaction and self-energy.
**`ibgw="1"`, `nbgw="8"`** | Computes quasiparticle corrections for bands 1 through 8.
**`skipgnd="false"`** | Recomputes Kohn-Sham states on the $GW$ k/q-point grid.
**`qdepw="sum"`** | Uses direct summation with the smearing parameter from **`freqgrid`**.
**`coreflag="xal"`** | Uses core and valence states in exchange, but only valence states in correlation.
**`nomeg="4"`** | Uses a very small frequency grid for speed.

<div style="clear: both;"></div>

The **`mixbasis`**, **`barecoul`**, **`scrcoul`**, and **`selfenergy`** elements control the mixed-product basis, the Coulomb interaction, screening, and the self-energy method. Their values are intentionally coarse in this example.


Save the notebook if you changed the XML block. The next cell keeps a copy of the ground-state input and writes the $GW$ input to **`input.xml`**.


In [ ]:
from pathlib import Path
import shutil

run_dir = Path("run_Si_taskGW")
current_tree = ET.parse(run_dir / "input.xml")
current_gs = current_tree.getroot().find("./groundstate")

if current_gs is not None and current_gs.get("do") == "fromscratch":
    shutil.copy2(run_dir / "input.xml", run_dir / "input_groundstate.xml")

input_str = get_input_xml_from_notebook(
    "task_based_gw",
    "SI_TASK_GW_FULL",
)
ET.fromstring(input_str)

with open(run_dir / "input.xml", "w") as fid:
    fid.write(input_str)


In [ ]:
%%bash
cd run_Si_taskGW
python3 -m excitingscripts.setup.excitingroot
cd ..


**<span style="color:#15317E">iii) Run the task group</span>**

This is the slowest step in the tutorial. All task files are written to **`run_Si_taskGW`**.


In [ ]:
%%bash
cd run_Si_taskGW
time python3 -m excitingscripts.execute.single -f input.xml
cd ..


<a id="Restart"></a>


<hr style="border:1px solid #DDD"> </hr>

### <span style="color:#15317E">4. Checking the Result and Restarting</span>

**<span style="color:#15317E">i) Check the task files</span>**

A successful direct run should produce the main $GW$ log, one or more files from every task, and the final quasiparticle file.


In [ ]:
%%bash
cd run_Si_taskGW

ls -1 GW_INFO.OUT VXCNN.OUT EVALQP.DAT EVALQP.OUT
ls -1 BARC_Q*.OUT SGI_Q*.OUT \
      EPSILON-GW_Q*.OUT INVERSE-EPSILON_Q*.OUT \
      SIGMAX_K*.OUT SIGMAC_K*.OUT | head -n 40

cd ..


The **`QPEigenvalues`** task always writes both outputs: **`EVALQP.DAT`** is readable text, while **`EVALQP.OUT`** is binary and should not be inspected with commands such as **`head`**. The task-group **`outputFormat`** selects the format of the reusable intermediate files that **`QPEigenvalues`** reads.

**<span style="color:#15317E">ii) Inspect the main $GW$ log</span>**


In [ ]:
%%bash
cd run_Si_taskGW
printf "Task entries in GW_INFO.OUT:\n"
grep -n "task:" GW_INFO.OUT || true
printf "\nLast lines of GW_INFO.OUT:\n"
tail -n 25 GW_INFO.OUT
cd ..


The log should contain the tasks **`vxc`**, **`Coulomb`**, **`sigmax`**, **`epsilon`**, **`invertEpsilon`**, **`sigmac`**, and **`QPEigenvalues`**.

**<span style="color:#15317E">iii) Restart only `QPEigenvalues`</span>**

The final task can be repeated without recomputing screening or self-energies when these files are present:

- **`VXCNN.OUT`** or **`VXCNN.DAT`**;
- **`SIGMAX_K*.OUT`**;
- **`SIGMAC_K*.OUT`**; and
- the DFT ground-state files.

A restart file is valid only when all reused files were produced with compatible grids, band ranges, basis settings, and numbers of empty states.

The task group in the restart input contains only

```xml
<taskGroup outputFormat="binary">
   <QPEigenvalues>
      <kpoints ki="1" kf="-1"/>
   </QPEigenvalues>
</taskGroup>
```


In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path

source = Path("run_Si_taskGW/input.xml")
target = Path("run_Si_taskGW/input_qp_restart.xml")

tree = ET.parse(source)
root = tree.getroot()
task_group = root.find("./gw/taskGroup")

for task in list(task_group):
    task_group.remove(task)

qp = ET.SubElement(task_group, "QPEigenvalues")
ET.SubElement(qp, "kpoints", {"ki": "1", "kf": "-1"})

ET.indent(tree, space="   ")
tree.write(target, encoding="unicode")
print(f"Wrote {target}")


The next cell temporarily replaces **`input.xml`** with the restart input. It restores the full task-group input when the command ends.


In [ ]:
%%bash
set -e
(
   cd run_Si_taskGW
   cp input.xml input_full_taskgroup.xml
   trap 'mv -f input_full_taskgroup.xml input.xml' EXIT
   cp input_qp_restart.xml input.xml
   time python3 -m excitingscripts.execute.single -f input.xml
)


**<span style="color:#15317E">iv) Common problems</span>**

problem | first check
:--- | :---
The XML helper cannot find a block. | Save the notebook and confirm that its file name is **`task_based_gw.ipynb`**.
A ground-state file is missing. | Read the last lines of **`INFO.OUT`** and fix the ground-state run first.
**`epsilon`** or **`invertEpsilon`** fails. | Check that the Coulomb files and the required dielectric files are present.
**`sigmac`** fails. | Check **`BARC_Q*.OUT`**, **`SGI_Q*.OUT`**, and **`INVERSE-EPSILON_Q*.OUT`**.
**`QPEigenvalues`** fails. | Check **`VXCNN.*`**, **`SIGMAX_K*.OUT`**, and **`SIGMAC_K*.OUT`**.
Results seem to come from an older calculation. | Start with a new or empty run directory.

<div style="clear: both;"></div>


<a id="Optional"></a>


<hr style="border:1px solid #DDD"> </hr>

### <span style="color:#15317E">5. Optional Workflow Branches</span>

The direct route is the easiest route to learn. Add one of the following branches only after the direct calculation works.

**<span style="color:#15317E">i) Store the polarizability</span>**

Add a **`polarizability`** task and tell **`epsilon`** to read the stored files:

```xml
<polarizability>
   <qpoints qi="1" qf="-1"/>
</polarizability>

<epsilon buildFromPolarizability="true">
   <qpoints qi="1" qf="-1"/>
</epsilon>
```

This route writes **`POLARIZABILITY-GW_Q*.OUT`** before **`EPSILON-GW_Q*.OUT`**.

**<span style="color:#15317E">ii) Use the irreducible q-point wedge</span>**

For a crystal, the dielectric matrices can be calculated on irreducible q-points and mapped back to the full grid:

```xml
<epsilon usingIrreducibleWedge="true">
   <qpoints qi="1" qf="-1"/>
</epsilon>

<invertEpsilon usingIrreducibleWedge="true">
   <qpoints qi="1" qf="-1"/>
</invertEpsilon>

<irreducibleMapping>
   <qpoints qi="1" qf="-1"/>
</irreducibleMapping>
```

The mapping task creates the full-grid **`INVERSE-EPSILON_Q*.OUT`** files needed by **`sigmac`**.

This tutorial uses **`qdepw="sum"`**. When the tetrahedron method **`qdepw="tet"`** is combined with the irreducible wedge, set **`enforceCrystalSymmetryTetrahedron="true"`** on the **`gw`** element.


<a id="Summary"></a>


<hr style="border:1px solid #DDD"> </hr>

### <span style="color:#15317E">6. Summary</span>

The task-based workflow used in this tutorial is:

1. run and check the DFT ground state;
2. run the seven tasks in **`taskGroup`**;
3. check **`GW_INFO.OUT`** and the task files;
4. keep compatible intermediate files; and
5. rerun only the task that needs to be repeated.

The central dependencies are

```text
Coulomb → epsilon → invertEpsilon → sigmac
Coulomb → sigmax
vxc + sigmax + sigmac → QPEigenvalues
```

Before using the quasiparticle energies in scientific work, test convergence with respect to the k/q-point grids, the number of empty states, the mixed-product basis, the Coulomb-basis cutoff, and the frequency grid.


<a id="Literature"></a>


<hr style="border:1px solid #DDD"> </hr>

### <span style="color:#15317E">Literature</span>

<a id="Hedin1965"></a> **<span style="color:firebrick">Hedin1965</span>** L. Hedin, *New Method for Calculating the One-Particle Green's Function with Application to the Electron-Gas Problem*, Phys. Rev. **139**, A796 (1965).

<a id="Hybertsen1986"></a> **<span style="color:firebrick">Hybertsen1986</span>** M. S. Hybertsen and S. G. Louie, *Electron correlation in semiconductors and insulators: Band gaps and quasiparticle energies*, Phys. Rev. B **34**, 5390 (1986).

<a id="Gulans2014"></a> **<span style="color:firebrick">Gulans2014</span>** A. Gulans, S. Kontur, C. Meisenbichler, D. Nabok, P. Pavone, S. Rigamonti, S. Sagmeister, U. Werner, and C. Draxl, *exciting: a full-potential all-electron package implementing density-functional theory and many-body perturbation theory*, J. Phys.: Condens. Matter **26**, 363202 (2014).

[**`exciting`** $GW$ input reference](https://exciting-code.org/home/about/input-reference/gw)

<hr style="border:2px solid #DDD"> </hr>
